In [1]:
!pip install -U accelerate transformers

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from google.colab import drive
import os
import shutil
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

In [3]:
# Desativa logs externos para evitar travar o notebook pedindo login
os.environ["WANDB_DISABLED"] = "true"

drive.mount('/content/drive')

# Define os caminhos baseados no diretório encontrado
BASE_PATH = '/content/drive/My Drive/Emotions/Dataset'

TRAIN_PATH = os.path.join(BASE_PATH, 'train_balanceado_aug.csv')
DEV_PATH = os.path.join(BASE_PATH, 'dev.csv')
TEST_PATH = os.path.join(BASE_PATH, 'test.csv')

# --- CONFIGURAÇÕES DO MODELO ---
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
MAX_LEN = 128
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 5
LEARNING_RATE = 1e-5

label_cols = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

Mounted at /content/drive


In [4]:
# --- CARREGAMENTO DE DADOS ---
def load_labeled_data(path):
    df = pd.read_csv(path)
    df['labels'] = df[label_cols].values.tolist()
    return df

print("Carregando arquivos...")
train_df = load_labeled_data(TRAIN_PATH)
val_df = load_labeled_data(DEV_PATH)
test_df = pd.read_csv(TEST_PATH)

# Preparar Teste
for col in label_cols:
    test_df[col] = 0.0
test_df['labels'] = test_df[label_cols].values.tolist()

Carregando arquivos...


In [5]:
# Essa função matemática força o modelo a aprender o que é difícil
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, logits=True, reduce=True):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.logits = logits
        self.reduce = reduce

    def forward(self, inputs, targets):
        if self.logits:
            BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        else:
            BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')

        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduce:
            return torch.mean(F_loss)
        else:
            return F_loss

In [6]:
# --- FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, logits=True, reduce=True):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.logits = logits
        self.reduce = reduce

    def forward(self, inputs, targets):
        if self.logits:
            BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        else:
            BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')

        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduce:
            return torch.mean(F_loss)
        else:
            return F_loss

In [7]:
# --- TOKENIZER E DATASET ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = dataframe.labels
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())
        inputs = self.tokenizer.encode_plus(
            text, None, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', return_token_type_ids=True, truncation=True
        )
        return {
            'input_ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(inputs['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(self.targets[index], dtype=torch.float)
        }

training_set = EmotionDataset(train_df, tokenizer, MAX_LEN)
validation_set = EmotionDataset(val_df, tokenizer, MAX_LEN)
testing_set = EmotionDataset(test_df, tokenizer, MAX_LEN)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
# --- TRAINER PERSONALIZADO ---
class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Focal Loss
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [9]:
# --- MODELO E ARGUMENTOS ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_cols), problem_type="multi_label_classification"
)

# O output_dir no Kaggle deve ser local (dentro do container)
training_args = TrainingArguments(
    output_dir="./results_large",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none" # IMPORTANTE: Evita erro de login no WandB
)

def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(preds))
    y_pred = np.zeros(probs.shape)
    y_pred[probs >= 0.4] = 1
    y_true = p.label_ids
    return {
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'accuracy': accuracy_score(y_true, y_pred)
    }

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=training_set,
    eval_dataset=validation_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-505838566.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTrainer.__init__`. Use `processing_class` instead.
  trainer = FocalTrainer(


In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,No log,0.072661,0.356411,0.415000
2,No log,0.062833,0.501539,0.490000
3,0.081600,0.066313,0.525181,0.455000
4,0.081600,0.065434,0.527171,0.460000
5,0.081600,0.067381,0.520740,0.460000


TrainOutput(global_step=905, training_loss=0.06691367428605728, metrics={'train_runtime': 728.6558, 'train_samples_per_second': 19.872, 'train_steps_per_second': 1.242, 'total_flos': 3373637129871360.0, 'train_loss': 0.06691367428605728, 'epoch': 5.0})

In [11]:
# --- PREDIÇÃO E OTIMIZAÇÃO ---
print("Calculando Thresholds...")
raw_pred, _, _ = trainer.predict(validation_set)
probs_dev = torch.sigmoid(torch.tensor(raw_pred)).numpy()
true_labels = val_df[label_cols].values

best_thresholds = {}
for i, col in enumerate(label_cols):
    best_score = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.95, 0.05):
        y_pred_temp = (probs_dev[:, i] >= thresh).astype(int)
        score = f1_score(true_labels[:, i], y_pred_temp)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    best_thresholds[col] = best_thresh
    print(f"{col}: {best_thresh:.2f} (F1: {best_score:.4f})")

Calculando Thresholds...


anger: 0.45 (F1: 0.7563)
disgust: 0.35 (F1: 0.2308)
fear: 0.35 (F1: 0.4000)
joy: 0.30 (F1: 0.7647)
sadness: 0.45 (F1: 0.5926)
surprise: 0.40 (F1: 0.6809)


# Savepoint da Nicole
- Alterações que fiz com relação ao uso dos thresholds

In [12]:
# tresholds
def apply_thresholds(probs, thresholds):
    thr = np.array([thresholds[c] for c in label_cols])
    return (probs >= thr).astype(int)

# inicialização
thresholds = {c: 0.5 for c in label_cols}
best_macro = 0.0

for round_ in range(3):  # 2–3 rodadas
    print(f"\n=== Rodada {round_+1} ===")
    for c in label_cols:
        best_t = thresholds[c]
        for t in np.arange(0.1, 0.91, 0.05):
            tmp = thresholds.copy()
            tmp[c] = t

            y_pred = apply_thresholds(probs_dev, tmp)
            macro = f1_score(true_labels, y_pred, average='macro', zero_division=0)

            if macro > best_macro:
                best_macro = macro
                best_t = t

        thresholds[c] = best_t
        print(f"{c}: threshold={best_t:.2f}")

print("\n✅ Melhor F1-Macro no DEV:", best_macro)
print("🎯 Thresholds finais:", thresholds)



=== Rodada 1 ===
anger: threshold=0.45
disgust: threshold=0.35
fear: threshold=0.35
joy: threshold=0.30
sadness: threshold=0.45
surprise: threshold=0.40

=== Rodada 2 ===
anger: threshold=0.45
disgust: threshold=0.35
fear: threshold=0.35
joy: threshold=0.30
sadness: threshold=0.45
surprise: threshold=0.40

=== Rodada 3 ===
anger: threshold=0.45
disgust: threshold=0.35
fear: threshold=0.35
joy: threshold=0.30
sadness: threshold=0.45
surprise: threshold=0.40

✅ Melhor F1-Macro no DEV: 0.5708702150921592
🎯 Thresholds finais: {'anger': np.float64(0.45000000000000007), 'disgust': np.float64(0.3500000000000001), 'fear': np.float64(0.3500000000000001), 'joy': np.float64(0.30000000000000004), 'sadness': np.float64(0.45000000000000007), 'surprise': np.float64(0.40000000000000013)}


In [15]:
# --- GERAÇÃO DO ARQUIVO FINAL (CORRIGIDA) ---
predictions_output = trainer.predict(testing_set)

# predictions_output.predictions já costuma ser numpy
probs_test = torch.sigmoid(torch.tensor(predictions_output.predictions)).cpu().numpy()

y_pred_final = np.zeros(probs_test.shape, dtype=int)

# ✅ APLICAR THRESHOLDS CORRETOS (macro-global do DEV)
# (garanta que a célula que calcula "thresholds" foi executada antes!)
print("Thresholds usados:", thresholds)

for i, col in enumerate(label_cols):
    t = thresholds[col]                     # <-- CORREÇÃO AQUI
    y_pred_final[:, i] = (probs_test[:, i] >= t).astype(int)

# Criar CSV
submission_df = pd.DataFrame(y_pred_final, columns=label_cols).astype(int)
submission_df["id"] = test_df["id"]
submission_df = submission_df[["id"] + label_cols]

OUTPUT_FILENAME = "submissao_corrigida.csv"
submission_df.to_csv(OUTPUT_FILENAME, index=False)

# Sanity checks
print("✅ Arquivo salvo:", OUTPUT_FILENAME)
print("Qtd de 1s por classe:\n", submission_df[label_cols].sum().sort_values(ascending=False))
print("All-zero:", (submission_df[label_cols].sum(axis=1) == 0).sum())

submission_df.head()

Thresholds usados: {'anger': np.float64(0.45000000000000007), 'disgust': np.float64(0.3500000000000001), 'fear': np.float64(0.3500000000000001), 'joy': np.float64(0.30000000000000004), 'sadness': np.float64(0.45000000000000007), 'surprise': np.float64(0.40000000000000013)}
✅ Arquivo salvo: submissao_corrigida.csv
Qtd de 1s por classe:
 joy         1540
anger       1438
sadness      648
disgust      562
fear         508
surprise     320
dtype: int64
All-zero: 782


,id,anger,disgust,fear,joy,sadness,surprise
0,ptbr_test_track_a_00001,1,0,0,0,1,0
1,ptbr_test_track_a_00002,0,0,0,1,0,0
2,ptbr_test_track_a_00003,1,0,0,0,0,0
3,ptbr_test_track_a_00004,0,0,0,0,0,0
4,ptbr_test_track_a_00005,0,0,0,1,0,0


## Checagem rápida para ver se está ok

In [16]:
print("Qtd de 1s por classe no TEST:")
print(submission_df[label_cols].sum().sort_values(ascending=False))

print("\nLinhas all-zero no TEST:")
print((submission_df[label_cols].sum(axis=1) == 0).sum())

Qtd de 1s por classe no TEST:
joy         1540
anger       1438
sadness      648
disgust      562
fear         508
surprise     320
dtype: int64

Linhas all-zero no TEST:
782
